# 第08回 ソルバ2：輸送計画と簡易ドッキング

 **今回はオンデマンド授業です。小レポートはありません。** 

スライドPDFとノートブックを並べ、上から順に進めてください。練習の解答は、このノートブックの末尾にまとめています。

1. 例題のコードを実行し、出力を確認します。
2. 「やってみよう」では結果を予想し、指定された行を変更して実行します。
3. 自分で試してから、末尾の解答と比べます。

コードを変更したら、 **そのコードセル全体を再実行** してください。画面に残っている出力は、前に実行したときの結果です。



<br>

## 1. Colabの準備

Googleアカウントでログインし、 **「ファイル → ドライブにコピーを保存」** を選びます。保存した自分のコピーで作業してください。

下の2セルを上から順に実行します。1つ目はインストール、2つ目はライブラリの読み込みと確認です。 **PuLP 3.3.0**  と、使えるソルバの一覧に  **HiGHS**  が表示されれば準備できています。

CPUで実行できます。新しい実行環境につながったときは、この準備からやり直します。


In [ ]:
# 線形計画を書くPuLPと、最適解を計算するHiGHSをインストールします。
%pip -q install "pulp==3.3.0" "highspy==1.12.0"


In [ ]:
# PuLPを読み込み、版と使えるソルバを確認します。
import pulp

print("PuLP", pulp.__version__)
print("使えるソルバ:", pulp.listSolvers(onlyAvailable=True))



<br>

## 2. 工場の生産計画（前回の復習）

製品1・2の生産量を $x_1,x_2$ とします。生産量は0以上で、小数も取れます。利益の単位は万円です。

$$\begin{aligned}
\max\quad z &= 4x_1+5x_2 \\
x_1+2x_2 &\le 5 &&\text{（原料A）}\\
2x_1+2x_2 &\le 8 &&\text{（原料B）}\\
x_1+x_2 &\le 10 &&\text{（原料C）}\\
x_1,x_2 &\ge 0
\end{aligned}$$

下のセルには、問題の作成から結果の表示までをまとめています。 **セル全体を実行** してください。


In [ ]:
# 1. 利益を最大にする問題を作ります。
model = pulp.LpProblem("Production", pulp.LpMaximize)

# 2. 生産量を表す変数を作ります。lowBound=0 は下限が0という指定です。
x1 = pulp.LpVariable("x1", lowBound=0)
x2 = pulp.LpVariable("x2", lowBound=0)

# 3. 目的関数（利益）と、原料の在庫による制約を追加します。
model += 4*x1 + 5*x2, "profit"
model += x1 + 2*x2 <= 5, "material_A"
model += 2*x1 + 2*x2 <= 8, "material_B"
model += x1 + x2 <= 10, "material_C"

# 4. HiGHSで解き、計算状態を確認します。
model.solve(pulp.HiGHS(msg=False))
factory_status = pulp.LpStatus[model.status]
print("状態:", factory_status)

# 5. 最適解が得られたときだけ、生産量と利益を表示します。
if factory_status == "Optimal":
    print("製品1の生産量:", pulp.value(x1))
    print("製品2の生産量:", pulp.value(x2))
    print("利益（万円）:", pulp.value(model.objective))
else:
    print("最適解を確認できません。計算状態と制約を確認してください。")



<br>

 **出力の確認** ：状態は `Optimal`、製品1は3、製品2は1、利益は17万円です。

生産量の組 $(3,1)$ が **最適解** 、利益17万円が **最適値** です。



<br>

## 3. 条件分岐：if・elif・else

条件を上から調べ、 **最初に成立した条件の処理だけ** を実行します。どの条件も成立しなければ `else` の処理を実行します。

`=` は右側の値を左側の変数に代入します。`==` は2つの値が等しいかを比較します。`if`・`elif`・`else` の行末には `:` を付け、その下の処理を半角スペース4個で字下げします。


In [ ]:
supply_total = 15  # 供給できる量の合計（t）
demand_total = 13  # 必要な量の合計（t）

if supply_total < demand_total:
    print("供給不足")
elif supply_total == demand_total:
    print("合計が一致")
else:
    print("供給に余裕")



<br>

### やってみよう：条件分岐

供給の合計は15のまま、需要の合計を  **13、15、17**  の順に変えます。

1. それぞれの場合に何と表示されるか、予想してください。
2. 上のセルの `demand_total = 13` の数値だけを変更します。
3.  **上のセル全体を実行** して、表示と予想を比べてください。

解答は末尾の「解答：条件分岐」にあります。



<br>

## 4. 輸送問題のデータ

2つの倉庫から3つの消費地へ荷物を運びます。 **必要な量を届け、供給上限を超えずに、総費用を最小にする** 輸送計画を求めます。

数量はt（トン）、単位費用は千円/tです。輸送量は小数も取れます。すべての経路を使え、費用は輸送量に比例するものとします。

| 出発地 | 消費地0への単位費用 | 消費地1への単位費用 | 消費地2への単位費用 | 供給上限（t） |
|:---|---:|---:|---:|---:|
| 倉庫0 | 2 | 5 | 6 | 7 |
| 倉庫1 | 4 | 1 | 3 | 8 |
| 需要（t） | 4 | 3 | 6 |  |

 **添字の順序は「倉庫、消費地」** です。倉庫の番号は0・1、消費地の番号は0・1・2です。



<br>

### 二重リストと添字

`cost` は、費用表の各行をリストにしたものです。`cost[0]` が倉庫0の行、`cost[0][2]` がその行の消費地2への単位費用です。

下のセルを実行すると、6が表示されます。


In [ ]:
# 数量はt、単位費用は千円/tです。
supply = [7, 8]       # 倉庫0・1の供給上限
demand = [4, 3, 6]    # 消費地0・1・2の需要
cost = [[2, 5, 6],    # 倉庫0から各消費地へ送る単位費用
        [4, 1, 3]]   # 倉庫1から各消費地へ送る単位費用

print(cost[0][2])



<br>

### 二重ループ

`range(2)` は0・1、`range(3)` は0・1・2です。外側の `i` で倉庫を選び、内側の `j` で消費地を順に選びます。

 **倉庫0の3経路、倉庫1の3経路** の順に表示されることを確認してください。


In [ ]:
for i in range(2):          # 倉庫を1つ選びます。
    for j in range(3):      # その倉庫から3つの消費地を順に調べます。
        print("倉庫", i, "消費地", j, "単位費用", cost[i][j])



<br>

### forの中での条件分岐

次は、単位費用が3以下の経路だけを表示します。`print` を `if` の下でさらに字下げします。

表示されるのは「倉庫0→消費地0」「倉庫1→消費地1」「倉庫1→消費地2」です。 **表示する経路を選ぶ処理** なので、このあと作る輸送問題では6経路すべてを使えます。


In [ ]:
for i in range(2):
    for j in range(3):
        if cost[i][j] <= 3:  # 単位費用が3以下のときだけ表示します。
            print("倉庫", i, "消費地", j, "単位費用", cost[i][j])



<br>

### やってみよう1：費用表の添字

費用表は `cost = [[2, 5, 6], [4, 1, 3]]` です。最初の行が倉庫0、次の行が倉庫1で、列は消費地0・1・2の順です。

 **倉庫1から消費地0への単位費用** を表示するコードを、下のコメントの次の行に1行書いてください。`print(cost[ ][ ])` の2つの空欄に整数を入れる形です。

値を予想してから実行し、末尾の「解答1」と比べてください。


In [ ]:
cost = [[2, 5, 6], [4, 1, 3]]
# この下に、指定された単位費用をprintで表示する1行を書いてください。



<br>

## 5. 輸送問題の目的関数と制約

$x_{ij}$ は倉庫 $i$ から消費地 $j$ へ送る量（t）、$c_{ij}$ はその経路の単位費用（千円/t）です。供給上限は $a=(7,8)$、需要は $b=(4,3,6)$ です。

 **目的関数：全6経路の費用を合計して最小化します。** 

$$\min\quad z=\sum_{i=0}^{1}\sum_{j=0}^{2}c_{ij}x_{ij}$$

$$z=2x_{00}+5x_{01}+6x_{02}+4x_{10}+x_{11}+3x_{12}$$

 **供給制約：各倉庫から送る量の合計を、供給上限以下にします。** 

$$\sum_{j=0}^{2}x_{ij}\le a_i\quad(i=0,1)$$

例えば倉庫0では、$x_{00}+x_{01}+x_{02}\le7$ です。

 **需要制約：各消費地への到着量の合計を、必要な量以上にします。** 

$$\sum_{i=0}^{1}x_{ij}\ge b_j\quad(j=0,1,2)$$

例えば消費地0では、$x_{00}+x_{10}\ge4$ です。輸送量はすべて $x_{ij}\ge0$ とします。

今回の単位費用はすべて正です。必要量より多く運ぶと費用が増えるので、最適解では到着量と需要が一致します。



<br>

## 6. PuLPによる輸送計画

### 内包表記とlpSum

内包表記は、繰り返しを使ってリストを作る書き方です。まず、数値の例で形を確認します。

```python
[j for j in range(3)]
# [0, 1, 2]

[[10*i+j for j in range(3)] for i in range(2)]
# [[0, 1, 2], [10, 11, 12]]
```

内側の `j` で1行を作り、外側の `i` で2行を並べています。次のセルでは数値の代わりにPuLPの変数を作ります。

- `f"x_{i}_{j}"` は、添字を変数の名前に入れる書き方です。`i=1, j=2` なら名前は `x_1_2` です。
- `pulp.lpSum(...)` は、複数の式を足し合わせます。
- `LpMinimize` は最小化、`lowBound=0` は変数の下限が0という指定です。


<br>

### 輸送問題の作成と計算

供給は `[7, 8]`、需要は `[4, 3, 6]`、費用表は `[[2, 5, 6], [4, 1, 3]]` です。

 **次のセル全体を実行** してください。データ、変数、目的関数、制約、計算の順に並んでいます。


In [ ]:
# 1. 問題のデータ（数量はt、単位費用は千円/t）
supply = [7, 8]
demand = [4, 3, 6]
cost = [[2, 5, 6], [4, 1, 3]]

# 2. 最小化問題と、2行3列の輸送量の変数を作ります。
transport = pulp.LpProblem("Transport", pulp.LpMinimize)
x = [[pulp.LpVariable(f"x_{i}_{j}", lowBound=0)
      for j in range(3)] for i in range(2)]

# 3. 全6経路の「単位費用 × 輸送量」を足します。
transport += pulp.lpSum(
    cost[i][j] * x[i][j]
    for i in range(2) for j in range(3)
)

# 4. 各倉庫の発送量（行の和）を供給上限以下にします。
for i in range(2):
    transport += pulp.lpSum(x[i][j] for j in range(3)) <= supply[i]

# 5. 各消費地への到着量（列の和）を需要以上にします。
for j in range(3):
    transport += pulp.lpSum(x[i][j] for i in range(2)) >= demand[j]

# 6. HiGHSで解きます。Optimalなら最適解が得られています。
transport.solve(pulp.HiGHS(msg=False))
transport_status = pulp.LpStatus[transport.status]
print("状態:", transport_status)



<br>

### 輸送量・費用・制約の確認

 **直前の「輸送問題の作成と計算」を実行してから** 、下のセルを実行します。計算状態が `Optimal` のときだけ、結果を表示します。

`pulp.value(x[i][j])` は、ソルバが求めた各経路の輸送量です。各倉庫の発送量と残量、各消費地への到着量を、それぞれ元の条件と比べます。


In [ ]:
if transport_status == "Optimal":
    # 2行3列の変数から、計算された輸送量を取り出します。
    flows = [[pulp.value(x[i][j]) for j in range(3)] for i in range(2)]
    for i in range(2):
        print(f"倉庫{i}から [消費地0, 1, 2] への輸送量（t）:", flows[i])
        print("  発送量:", sum(flows[i]), "供給上限:", supply[i])
        print("  残量:", supply[i] - sum(flows[i]))

    # 各消費地への到着量を、必要な量と比べます。
    for j in range(3):
        arrived = sum(flows[i][j] for i in range(2))
        print(f"消費地{j}への到着量（t）:", arrived, "需要:", demand[j])

    print("総費用（千円）:", pulp.value(transport.objective))
else:
    print("最適解を確認できません。計算状態と供給・需要を確認してください。")



<br>

 **この例題の結果** は、倉庫0から $[4,0,1]$ t、倉庫1から $[0,3,5]$ t、総費用32千円です。

- 発送量は5tと8tで、供給上限7tと8t以下です。残量は2tと0tです。
- 到着量は4t・3t・6tで、各消費地の需要を満たします。
- 輸送量はすべて0以上です。

費用も元の式で確かめられます。

$$2\times4+6\times1+1\times3+3\times5=32\quad\text{（千円）}$$



<br>

## 7. 輸送計画のプロット

 **「輸送量・費用・制約の確認」のあとで** 、下のセルを実行します。倉庫と消費地を点で表し、輸送量が正の経路に矢印を描きます。

- `ax.scatter(...)`：点を描きます。青丸が倉庫、橙の四角が消費地です。
- `ax.annotate(...)`：矢印を描きます。`xytext` が出発点、`xy` が到着点です。
- `ax.text(...)`：位置と文字を指定し、輸送量を表示します。

図の `Warehouse` は倉庫、`City` は消費地です。点の位置は図を見やすくするための配置で、距離を費用計算には使っていません。

まず図を表示し、 **矢印の太さを指定する `lw=1+v*0.6` の `0.6` を `1.0` に変えて再実行** してみてください。輸送量に応じて太さが変わります。


In [ ]:
import matplotlib.pyplot as plt

if transport_status == "Optimal":
    fig, ax = plt.subplots(figsize=(8, 4.5))
    src = [(0, 2), (0, 0)]              # 倉庫を描く位置
    dst = [(4, 2.5), (4, 1), (4, -0.5)] # 消費地を描く位置

    # 倉庫を青丸、消費地を橙の四角で描きます。
    for i, (a, b) in enumerate(src):
        ax.scatter(a, b, s=240, c="#0017c1")
        ax.text(a-0.6, b+0.35, f"Warehouse {i}")
    for j, (a, b) in enumerate(dst):
        ax.scatter(a, b, s=200, c="#c06900", marker="s")
        ax.text(a+0.15, b, f"City {j}")

    # 正の輸送量がある経路に、矢印と輸送量を表示します。
    for i in range(2):
        for j in range(3):
            v = flows[i][j]
            if v > 1e-7:  # ごく小さい計算誤差は0とみなします。
                a, b = src[i]
                c, d = dst[j]
                ax.annotate("", xy=(c, d), xytext=(a, b),
                            arrowprops=dict(arrowstyle="->",
                                            lw=1+v*0.6, color="#16764b"))
                f = 0.32 if i == 0 else 0.68  # 文字が重ならない位置
                ax.text(a+(c-a)*f, b+(d-b)*f, f"{v:g} t",
                        bbox=dict(fc="white", ec="none"))

    ax.set(xlim=(-1, 5.2), ylim=(-1, 3.2),
           title=f"Transport cost = {pulp.value(transport.objective):g} (1000 yen)")
    ax.axis("off")
    plt.show()
else:
    print("輸送問題の状態を確認してください。Optimalのときに図を描きます。")



<br>

### やってみよう2：需要の変更

消費地2の需要を  **6tから10tへ** 増やします。供給上限は `[7, 8]`、ほかの消費地の需要は4t・3tのままです。

1. 供給と需要の合計を比べ、すべての需要を満たせるか予想してください。
2. 下のセルの `changed_demand = [4, 3, 6]` の  **最後の6だけを10へ変更** します。
3.  **このセル全体を実行** し、表示される計算状態を確認してください。

元の例題とは別の問題を作るので、例題のデータを戻す操作は不要です。解答は末尾の「解答2」にあります。


In [ ]:
# この行の最後の6を10へ変更します。
changed_demand = [4, 3, 6]
changed_supply = [7, 8]
changed_cost = [[2, 5, 6], [4, 1, 3]]

# 変更後の需要を使って、新しい輸送問題を作ります。
changed = pulp.LpProblem("Transport_changed_demand", pulp.LpMinimize)
changed_x = [[pulp.LpVariable(f"changed_x_{i}_{j}", lowBound=0)
              for j in range(3)] for i in range(2)]
changed += pulp.lpSum(changed_cost[i][j] * changed_x[i][j]
                      for i in range(2) for j in range(3))
for i in range(2):
    changed += pulp.lpSum(changed_x[i][j] for j in range(3)) <= changed_supply[i]
for j in range(3):
    changed += pulp.lpSum(changed_x[i][j] for i in range(2)) >= changed_demand[j]

# 供給・需要の合計と、ソルバの計算状態を比べます。
print("供給の合計（t）:", sum(changed_supply))
print("需要の合計（t）:", sum(changed_demand))
changed.solve(pulp.HiGHS(msg=False))
changed_status = pulp.LpStatus[changed.status]
print("状態:", changed_status)
if changed_status == "Optimal":
    print("最小費用（千円）:", pulp.value(changed.objective))
else:
    print("最適解を確認できません。供給と需要を確認してください。")



<br>

## 8. 簡易ドッキング：3点の配置

3点を形の変わらない分子に見立て、対応する目標点へ近づけます。 **点0・1・2の対応は固定** し、全点を同じ量だけ平行移動します。

| 点の番号 | 元の点の横座標 | 元の点の縦座標 | 目標点の横座標 | 目標点の縦座標 |
|:---|---:|---:|---:|---:|
| 0 | 0 | 0 | 3 | 2 |
| 1 | 2 | 0 | 3 | 4 |
| 2 | 2 | 1 | 2 | 4.5 |

回転は  **0°・90°・180°・270°の4候補** です。各候補について、目標点とのずれが最小になる平行移動を求めます。

結合エネルギーや立体衝突は含めません。ここで求めるのは、4候補の中で座標のずれが最小になる配置です。



<br>

### ずれの絶対値と補助変数

回転後の点を $(a_{ix},a_{iy})$、目標点を $(b_{ix},b_{iy})$、全点共通の平行移動を $(t_x,t_y)$ とします。

横のずれを $d=a_{ix}+t_x-b_{ix}$ とすると、次の2条件で $u_{ix}\ge |d|$ を表せます。

$$u_{ix}\ge d,\qquad u_{ix}\ge-d$$

例えば $d=-2$ では、$u_{ix}\ge-2$ と $u_{ix}\ge2$ なので、最小の $u_{ix}$ は2です。 **目的関数で $u_{ix}$ を最小化することで、最適解では $u_{ix}=|d|$ になります。**  縦方向も同じです。

1つの回転候補で解く問題は、次の線形計画です。

$$\begin{aligned}
\min\quad &\sum_{i=0}^{2}(u_{ix}+u_{iy}) \\
u_{ix}&\ge a_{ix}+t_x-b_{ix} \\
u_{ix}&\ge-(a_{ix}+t_x-b_{ix}) \\
u_{iy}&\ge a_{iy}+t_y-b_{iy} \\
u_{iy}&\ge-(a_{iy}+t_y-b_{iy}) \\
u_{ix},u_{iy}&\ge0\quad(i=0,1,2) \\
t_x,t_y&\in\mathbb{R}
\end{aligned}$$

移動量は左・下への移動も許すので、負も取れます。各候補の座標は定数です。回転角を連続変数にした場合は、この形の線形計画にはなりません。



<br>

### 4候補の計算

次のセルは、データの準備と4候補の計算です。 **まずそのままセル全体を実行** してください。その次の「4候補のプロット」で図を描きます。

NumPyは配列の計算に使います。`a = ligand @ rotation.T` は、用意した回転行列で3点の座標を回転させる計算です。4候補を同じ条件で比較します。

`target_x_shift = 0` が、あとで練習する変更箇所です。ここでは0のままにします。


In [ ]:
import numpy as np

# 元の3点と、対応する目標点の座標です。
ligand = np.array([[0, 0], [2, 0], [2, 1]], dtype=float)
target = np.array([[3, 2], [3, 4], [2, 4.5]], dtype=float)
target_x_shift = 0  # やってみよう3で、この0を変更します。
target[:, 0] += target_x_shift  # 全目標点の横座標へ同じ量を加えます。

# 原点を中心とした、0・90・180・270度の回転行列です。
rotations = [np.array([[1,0],[0,1]]), np.array([[0,-1],[1,0]]),
             np.array([[-1,0],[0,-1]]), np.array([[0,1],[-1,0]])]
# 4候補それぞれの最適な平行移動を求めます。
results = []
for angle, rotation in zip([0, 90, 180, 270], rotations):
    a = ligand @ rotation.T
    fit = pulp.LpProblem(f"Pose_{angle}", pulp.LpMinimize)
    # 平行移動は負も許します。lowBound=None が自由変数の指定です。
    tx = pulp.LpVariable("tx", lowBound=None)
    ty = pulp.LpVariable("ty", lowBound=None)
    # 各点の横・縦のずれの絶対値を、補助変数で表します。
    ux = [pulp.LpVariable(f"ux_{i}", lowBound=0) for i in range(3)]
    uy = [pulp.LpVariable(f"uy_{i}", lowBound=0) for i in range(3)]
    fit += pulp.lpSum(ux) + pulp.lpSum(uy)
    for i in range(3):
        fit += ux[i] >= a[i,0] + tx - target[i,0]
        fit += ux[i] >= -(a[i,0] + tx - target[i,0])
        fit += uy[i] >= a[i,1] + ty - target[i,1]
        fit += uy[i] >= -(a[i,1] + ty - target[i,1])
    # 計算後、元の座標でずれと点どうしの距離を検算します。
    fit.solve(pulp.HiGHS(msg=False))
    assert pulp.LpStatus[fit.status] == "Optimal", "計算状態を確認してください"
    shift = np.array([pulp.value(tx), pulp.value(ty)])
    pose = a + shift
    score = pulp.value(fit.objective)
    # 全点の平行移動が同じなので、点どうしの距離は変わりません。
    assert np.allclose(np.linalg.norm(pose[:,None]-pose[None,:],axis=-1),
                       np.linalg.norm(ligand[:,None]-ligand[None,:],axis=-1))
    assert np.isclose(score, np.abs(pose-target).sum())
    results.append(dict(angle=angle, shift=shift, pose=pose, score=score))
    print(f"回転 {angle:3}度: 移動 {shift}, ずれ合計 {score:.2f}")
best = min(results, key=lambda r: r["score"])
print("候補内で最良の回転:", best["angle"], "度")



<br>

### 4候補のプロット

 **直前の「4候補の計算」を実行してから** 、下のセルを実行します。

`ax.plot(...)` で移動後の3点を結び、`ax.scatter(...)` で目標点を描きます。橙の×印が目標点（Targets）、緑の丸と線が移動後の形（Rigid ligand）です。点の番号が対応を示します。

各図の `L1` は、横と縦のずれの絶対値を全点で合計した値です。4つの値を比べ、最小の候補を確認してください。


In [ ]:
import matplotlib.pyplot as plt

# 2行2列にして、各候補の図を大きく表示します。
fig, axes = plt.subplots(2, 2, figsize=(9, 9))
for ax, result in zip(axes.flat, results):
    pose = result["pose"]
    ax.scatter(*target.T, marker="x", s=90, c="#c06900", label="Targets")
    ax.plot(*pose.T, "o-", c="#16764b", lw=3, label="Rigid ligand")

    # 対応する点のずれを、灰色の点線で示します。
    for i in range(3):
        ax.plot([pose[i, 0], target[i, 0]],
                [pose[i, 1], target[i, 1]], ":", c="#888")
        ax.annotate(str(i), target[i], xytext=(6, 6), textcoords="offset points")

    ax.set(xlim=(-1+target_x_shift, 6+target_x_shift), ylim=(0, 7),
           title=f'{result["angle"]} deg / L1 = {result["score"]:.1f}',
           xlabel="x", ylabel="y", aspect="equal")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=9, loc="lower left")

plt.tight_layout()
plt.show()



<br>

### やってみよう3：目標点の平行移動

目標点は `(3, 2), (3, 4), (2, 4.5)` です。3点すべてを横方向へ1だけ動かします。元の3点と4つの回転候補は変えません。

1. 最良の回転候補、移動量、ずれの合計がどう変わるか予想してください。
2. 「4候補の計算」の  **`target_x_shift = 0` を `target_x_shift = 1` に変更** します。
3.  **「4候補の計算」→「4候補のプロット」** の順に、2つのセル全体を実行します。
4. 出力と図を予想と比べ、末尾の「解答3」で確認してください。

同じセルを繰り返し実行しても、指定した値を超えて目標点が動くことはありません。元へ戻すときは、1を0に戻して同じ2セルを実行します。



<br>

## 9. やってみよう4：輸送条件の変更（任意）

次の条件で輸送問題を解いてください。単位費用は例題と同じです。

| 出発地 | 消費地0への単位費用 | 消費地1への単位費用 | 消費地2への単位費用 | 供給上限（t） |
|:---|---:|---:|---:|---:|
| 倉庫0 | 2 | 5 | 6 | 8 |
| 倉庫1 | 4 | 1 | 3 | 9 |
| 需要（t） | 4 | 5 | 6 |  |

1. 下のセルの `trial_supply = None` の  **`None` を、2倉庫の供給上限のリストに置き換えます** 。
2. `trial_demand = None` の  **`None` を、3消費地の需要のリストに置き換えます** 。
3.  **セル全体を実行** し、状態、輸送量、総費用、供給の残量、各消費地への到着量を確認します。

リストは `[数値, 数値]` のように、半角の角括弧とカンマで書きます。解答は末尾の「解答4」にあります。


In [ ]:
# Noneの2か所を、問題文の数値を並べたリストに置き換えます。
trial_supply = None
trial_demand = None
trial_cost = [[2, 5, 6], [4, 1, 3]]  # 単位費用は変更しません。

if trial_supply is None or trial_demand is None:
    print("先頭のNoneを供給・需要のリストに置き換え、セル全体を実行してください。")
else:
    # 変更後のデータから、新しい最小化問題を作ります。
    trial = pulp.LpProblem("Transport_trial", pulp.LpMinimize)
    trial_x = [[pulp.LpVariable(f"trial_x_{i}_{j}", lowBound=0)
                for j in range(3)] for i in range(2)]
    trial += pulp.lpSum(trial_cost[i][j] * trial_x[i][j]
                        for i in range(2) for j in range(3))

    # 発送量は供給上限以下、到着量は需要以上にします。
    for i in range(2):
        trial += pulp.lpSum(trial_x[i][j] for j in range(3)) <= trial_supply[i]
    for j in range(3):
        trial += pulp.lpSum(trial_x[i][j] for i in range(2)) >= trial_demand[j]

    trial.solve(pulp.HiGHS(msg=False))
    trial_status = pulp.LpStatus[trial.status]
    print("状態:", trial_status)
    if trial_status == "Optimal":
        trial_flows = [[pulp.value(trial_x[i][j]) for j in range(3)]
                       for i in range(2)]
        for i in range(2):
            print(f"倉庫{i}からの輸送量（t）:", trial_flows[i])
            print("  残量（t）:", trial_supply[i] - sum(trial_flows[i]))
        for j in range(3):
            arrived = sum(trial_flows[i][j] for i in range(2))
            print(f"消費地{j}への到着量（t）:", arrived, "需要:", trial_demand[j])
        print("総費用（千円）:", pulp.value(trial.objective))
    else:
        print("最適解を確認できません。供給と需要を確認してください。")



<br>

## 10. 大規模な輸送問題：100倉庫・100消費地

小さな例題と同じ式で、 **輸送量1万個を決める問題** を解きます。データは授業用に作った架空のものです。

| 項目 | この演習の設定 |
|---|---|
| 倉庫・消費地 | 各100か所。番号はそれぞれ0〜99 |
| 位置 | 100 km × 100 kmの平面上。道路や地形は考えない |
| 供給上限 | 初期設定では各倉庫100 t、合計10,000 t |
| 需要 | 初期設定では各消費地60〜100 t、合計7,915 t |
| 単位費用 | 直線距離1 kmにつき10円/t。倉庫から消費地へ直接運ぶ |
| 輸送量 | 0以上の連続量（小数も可）。すべての経路を使える |

 **倉庫の供給上限を守り、各消費地の需要を満たす配送のうち、総費用が最も小さいものを求めます。** 車両の台数や巡回順序を決める問題ではありません。

この後の4つのコードセルは、 **①データ → ②順番に配送 → ③最適化 → ④図** の順で実行します。最初は値を書き換えずに試してください。

### ① データの準備

`demand_scale` は各消費地の需要に掛ける倍率、`supply_scale` は各倉庫の供給上限に掛ける倍率です。初期値はどちらも1.0です。

同じ乱数の初期値2026を使うので、再実行しても位置と基準の需要は変わりません。比較する条件だけを変えられます。`warehouse_xy[i, 0]` と `[i, 1]` は倉庫iの横・縦の座標です。`cost_big[i, j]` は倉庫iから消費地jへの単位費用です。

<br>


In [ ]:
# 変更するのは demand_scale と supply_scale の2行です。
# 初回は両方1.0のまま、このセルから④までを順に実行します。
demand_scale = 1.0
supply_scale = 1.0

import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter
import pulp

# 同じ位置・同じ基準の需要を毎回作り、条件の違いだけを比較します。
rng = np.random.default_rng(2026)
n_warehouse, n_market = 100, 100
warehouse_xy = rng.uniform(0, 100, size=(n_warehouse, 2))
market_xy = rng.uniform(0, 100, size=(n_market, 2))
base_demand = rng.integers(60, 101, size=n_market)
base_supply = np.full(n_warehouse, 100.0)
supply_big = base_supply * supply_scale
demand_big = base_demand * demand_scale

# 各経路の直線距離（km）を計算します。行が倉庫、列が消費地です。
cost_big = np.zeros((n_warehouse, n_market))
for i in range(n_warehouse):
    for j in range(n_market):
        dx = warehouse_xy[i, 0] - market_xy[j, 0]
        dy = warehouse_xy[i, 1] - market_xy[j, 1]
        cost_big[i, j] = 10 * np.sqrt(dx**2 + dy**2)  # 円/t

assert demand_scale > 0 and supply_scale >= 0, "需要は正、供給は0以上にしてください。"
print("架空の物流データ：100 km × 100 km の平面")
print("倉庫:", n_warehouse, "消費地:", n_market)
print("輸送量の変数:", n_warehouse * n_market)
print("供給・需要の制約:", n_warehouse + n_market, "本（非負条件は別）")
print(f"総供給上限: {supply_big.sum():,.1f} t")
print(f"総需要: {demand_big.sum():,.1f} t")
print("先頭5か所の需要（t）:", demand_big[:5])



<br>

### ② 近い倉庫から順に配送する方法

 **消費地0から99まで順番に** 配送を決めます。各消費地について、近い倉庫から在庫の範囲内で荷物を受け取り、足りなければ次に近い倉庫を使います。

先に決めた配送は変更しません。そのため、後の消費地が遠い倉庫から受け取ることがあります。消費地の順番によって結果も変わります。これは比較用の方法で、最小費用を保証するものではありません。

`greedy_ship[i, j]` はこの方法で決めた輸送量、`remaining[i]` は倉庫iの残量です。

<br>


In [ ]:
# 消費地0から99まで順番に、近い倉庫の在庫を使います。
# np.argsortは、単位費用が小さい順に倉庫の番号を並べます。
greedy_ship = np.zeros((n_warehouse, n_market))
remaining = supply_big.copy()  # 元の供給上限を残し、残量だけを更新します。
start = perf_counter()
for j in range(n_market):
    needed = demand_big[j]
    for i in np.argsort(cost_big[:, j]):
        amount = min(remaining[i], needed)
        greedy_ship[i, j] = amount
        remaining[i] -= amount
        needed -= amount
        if needed < 1e-8:
            break

greedy_seconds = perf_counter() - start
# axis=0は倉庫方向の合計です。各消費地への到着量を求めます。
greedy_ok = np.all(greedy_ship.sum(axis=0) >= demand_big - 1e-6)
if greedy_ok:
    greedy_cost = float(np.sum(cost_big * greedy_ship))
    print(f"近い倉庫から順に配送した費用: {greedy_cost:,.0f} 円")
    print(f"配送量の計算時間: {greedy_seconds:.3f} 秒")
else:
    greedy_cost = None
    print("供給が不足しています。すべての需要を満たす配送はできません。")


<br>

### ③ ソルバによる最適化

 **①で作った同じ位置・供給上限・需要・単位費用** を使い、すべての経路の輸送量をまとめて決めます。

倉庫と消費地の番号 $i,j$ は、それぞれ0〜99です。$x_{ij}$ は輸送量（t）、$c_{ij}$ は単位費用（円/t）、$s_i$ は供給上限（t）、$d_j$ は需要（t）です。

$$
\min\quad z=\sum_{i=0}^{99}\sum_{j=0}^{99}c_{ij}x_{ij}
$$

$$
\sum_{j=0}^{99}x_{ij}\leq s_i\quad(i=0,\ldots,99)
$$

$$
\sum_{i=0}^{99}x_{ij}\geq d_j\quad(j=0,\ldots,99),\qquad x_{ij}\geq0
$$

前の小さな輸送問題から変わったのは **データと添字の範囲** です。変数は10,000個、供給制約100本と需要制約100本です。非負条件は各変数の `lowBound=0` で指定します。

下のセル全体を実行します。PuLP（パルプ）でモデルを書き、HiGHSで解きます。 **初期設定と今回の設定を両方計算** するので、初期設定の結果は自分で保存する必要はありません。

**実行後は、このセルのすぐ下を確認します。**

| 出力の名前 | 確認すること |
|---|---|
| 状態 | `Optimal` なら最適解が得られています |
| 供給上限・需要・非負条件 | `True True True` なら3種類の条件をすべて満たしています |
| 今回の最小の総費用 | 初期設定の602,850円と比べます |
| 初期設定からの増減 | プラスなら高く、マイナスなら安くなっています |

「同じ条件で、順番に配送する方法より○円安い」は、**同じ需要・供給で2つの配送方法を比べた差** です。初期設定からの増減は、その1行上に表示します。

<br>


In [ ]:
# 初期設定と今回の設定を、同じ式でそれぞれ解きます。
# 初期設定の解も毎回作るので、実行順に左右されずに比較できます。
solved_plans = []
for supply, demand in [(base_supply, base_demand), (supply_big, demand_big)]:
    start = perf_counter()
    large_model = pulp.LpProblem("LargeTransport", pulp.LpMinimize)
    ship = [[pulp.LpVariable(f"ship_{i}_{j}", lowBound=0)
             for j in range(n_market)] for i in range(n_warehouse)]
    # 目的関数：全経路の「単位費用 × 輸送量」の合計です。
    large_model += pulp.lpSum(cost_big[i, j] * ship[i][j]
                             for i in range(n_warehouse)
                             for j in range(n_market))
    # 各倉庫の発送量は供給上限以下です。
    for i in range(n_warehouse):
        large_model += pulp.lpSum(ship[i][j] for j in range(n_market)) <= supply[i]
    # 各消費地への到着量は需要以上です。
    for j in range(n_market):
        large_model += pulp.lpSum(ship[i][j] for i in range(n_warehouse)) >= demand[j]
    model_seconds = perf_counter() - start
    start = perf_counter()
    large_model.solve(pulp.HiGHS(msg=False))
    solve_seconds = perf_counter() - start
    large_status = pulp.LpStatus[large_model.status]
    plan = None  # 解がないときに、前の結果を使わないようにします。
    if large_status == "Optimal":
        plan = np.array([[pulp.value(v) for v in row] for row in ship])
    solved_plans.append(plan)

# 1つ目が初期設定、2つ目が今回の設定の解です。
initial_ship, optimal_ship = solved_plans
initial_cost = float(np.sum(cost_big * initial_ship))
print("状態:", large_status)
if optimal_ship is not None:
    optimal_cost = float(np.sum(cost_big * optimal_ship))
    supply_ok = np.all(optimal_ship.sum(axis=1) <= supply_big + 1e-6)
    demand_ok = np.all(optimal_ship.sum(axis=0) >= demand_big - 1e-6)
    nonnegative_ok = np.all(optimal_ship >= -1e-6)
    print("供給上限・需要・非負条件:", supply_ok, demand_ok, nonnegative_ok)
    print(f"初期設定の最小の総費用: {initial_cost:,.0f} 円")
    print(f"今回の最小の総費用: {optimal_cost:,.0f} 円")
    # 表示した金額どうしの差です。プラスなら初期設定より高くなります。
    cost_change = round(optimal_cost) - round(initial_cost)
    print(f"初期設定からの増減: {cost_change:+,} 円")
    if greedy_ok:
        saving = greedy_cost - optimal_cost
        print(f"同じ条件で、順番に配送する方法より {saving:,.0f} 円安い"
              f"（{100*saving/greedy_cost:.1f}%）")
else:
    print("今回の条件では需要を満たす配送ができません。図は表示しません。")
print(f"今回の計算時間：モデル作成 {model_seconds:.3f} 秒、求解 {solve_seconds:.3f} 秒")


<br>

### ④ 配送経路の比較

下のセルを実行すると、**2枚の図** が出ます。

| 図 | 左 | 右 |
|---|---|---|
| 1枚目：配送方法の比較 | 今回の条件で、近い倉庫から順に配送 | 今回の条件で、費用を最小化 |
| 2枚目：条件変更の前後の比較 | 初期設定で、費用を最小化 | 今回の条件で、費用を最小化 |

**「やってみよう5」では2枚目を確認します。** 左は `Initial`（初期設定）、右は `Current`（今回の設定）です。最初に両方の倍率を1.0にして実行したときは、左右が同じになります。

青い四角は倉庫（Warehouse）、橙の丸は消費地（Market）です。**点の位置は変わりません。点どうしを結ぶ緑の経路や、線の太さがどう変わったかを比べてください。** 太い線ほど輸送量が多く、同じ輸送量ならどの図でも同じ太さです。凡例の20 t・60 t・100 tの線を目安にします。

輸送量が正の経路だけを表示します。図のx・yは位置の座標（km）で、輸送量 $x_{ij}$ とは別です。

<br>


In [ ]:
# 1枚目は配送方法、2枚目は初期設定と今回の設定を比べます。
from matplotlib.lines import Line2D
if optimal_ship is not None and greedy_ok:
    comparisons = [
        ("1. Methods under current conditions", [greedy_ship, optimal_ship],
         [f"Nearest first: {greedy_cost:,.0f} yen", f"Minimum cost: {optimal_cost:,.0f} yen"]),
        ("2. Before and after changing conditions", [initial_ship, optimal_ship],
         [f"Initial: {initial_cost:,.0f} yen", f"Current: {optimal_cost:,.0f} yen"]),
    ]
    for heading, plans, titles in comparisons:
        fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)
        for ax, plan, title in zip(axes, plans, titles):
            for i, j in np.argwhere(plan > 1e-6):
                # 基準を100 tに固定し、条件を変えても同じ太さで比べます。
                ax.plot([warehouse_xy[i, 0], market_xy[j, 0]],
                        [warehouse_xy[i, 1], market_xy[j, 1]],
                        color="seagreen", alpha=0.6,
                        linewidth=0.3 + 2.2 * plan[i, j] / 100)
            ax.scatter(warehouse_xy[:, 0], warehouse_xy[:, 1],
                       marker="s", s=25, color="royalblue", label="Warehouse")
            ax.scatter(market_xy[:, 0], market_xy[:, 1],
                       marker="o", s=18, color="darkorange", label="Market")
            ax.set(title=title, xlabel="x (km)", ylabel="y (km)")
            ax.set_xlim(-3, 103)
            ax.set_ylim(-3, 103)
            ax.set_aspect("equal")
        handles, labels = axes[0].get_legend_handles_labels()
        for amount in [20, 60, 100]:
            handles.append(Line2D([0], [0], color="seagreen",
                                  linewidth=0.3 + 2.2 * amount / 100))
            labels.append(f"{amount} t")
        fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.93), ncol=5)
        fig.suptitle(heading + "\nSynthetic teaching data (not actual logistics)")
        fig.tight_layout(rect=(0, 0, 1, 0.82))
        plt.show()
else:
    print("今回の条件では図を表示できません。③の状態と①の入力値を確認してください。")



<br>

初期設定では、順番に配送する方法が **約689,159円** 、最適化した総費用が **約602,850円** になります。この架空データでは約86,309円、12.5%下がります。データや条件を変えると、この割合も変わります。

先に近い倉庫の在庫を使い切ると、別の消費地が遠方から受け取る場合があります。ソルバは供給上限と需要を同時に考慮し、配送の組合せを調整します。

計算時間は実行環境で変わります。表示した「求解」は `solve` の呼出し時間で、データ作成・インストール・図の表示時間は含みません。

<br>


<br>

### やってみよう5：需要と供給を変える

100倉庫・100消費地の演習です。初期設定は、総供給上限10,000 t、総需要7,915 t、最小の総費用602,850円です。

**変更する場所：上の「① データの準備」のコードセル内にある、次の2行です。** それ以外の行はそのままにします。

```python
demand_scale = 1.0  # 各消費地の需要に掛ける倍率
supply_scale = 1.0  # 各倉庫の供給上限に掛ける倍率
```

| 試す条件 | demand_scale | supply_scale |
|---|---:|---:|
| A：需要を10%増やす | 1.1 | 1.0 |
| B：需要を元に戻し、供給上限を10%減らす | 1.0 | 0.9 |

**Aを試してからBを試します。毎回、①データ → ②順番に配送 → ③最適化 → ④図の4セルを、上から順に再実行してください。** ①の2行を変えただけでは、計算結果は更新されません。

費用が初期設定より高くなるか安くなるかを予想し、実行後に次を確認します。

| 確認する場所 | 確認すること |
|---|---|
| ①の下の「総供給上限」「総需要」 | Aでは需要だけ、Bでは供給上限だけが初期設定から変わったか |
| ③の下の「状態」 | `Optimal`（最適解が得られた）か |
| ③の下の「供給上限・需要・非負条件」 | `True`が3つ並んでいるか。3種類の条件を満たすという意味です |
| ③の下の「今回の最小の総費用」「初期設定からの増減」 | 602,850円からいくら増えたか、減ったか |
| ④の2枚目の図 | 左が初期設定、右が今回。配送経路と輸送量がどう変わったか |

「順番に配送する方法より○円安い」は、**同じ条件で配送方法を比べた差** です。初期設定からの増減と混同しないようにしてください。

確認できたら、末尾の「解答5」と比べます。小レポートはありません。

<br>



<br>

## まとめと次回

- 輸送問題では、各経路の輸送量を変数とし、費用の合計を最小化しました。各倉庫の発送量と各消費地への到着量を制約で指定しました。
- 大規模な輸送問題でも同じモデルを使い、100倉庫・100消費地の輸送量1万個を求めました。順番に近い倉庫から配送する方法と、費用・経路を比べました。
- `for` で複数の変数や制約を作り、`if` で条件に応じて処理を選びました。
- 簡易ドッキングでは、絶対値を補助変数と不等式で表し、4つの回転候補を比較しました。

次回は、途中の地点を通る **最短経路** を扱います。その次は、経路ごとの容量を守って流す量を最大にする **最大流** です。輸送費用、経路の長さ、流せる容量の違いを確認します。

 **今回は小レポートはありません。**  自分のDriveにノートブックが保存されていることを確認してください。

参考：[PuLP 3.3.0](https://github.com/coin-or/pulp/tree/3.3.0)、[HiGHS](https://highs.dev/)、[AutoDock Vina原著](https://pmc.ncbi.nlm.nih.gov/articles/PMC3041641/)。



<br>

## 練習の解答

自分で入力・実行してから、該当する解答を確認してください。



<br>

### 解答：条件分岐

供給の合計は15です。需要を13・15・17へ変えたときの結果は次のとおりです。

| 需要の合計 | 実行する処理 | 表示 |
|---:|:---|:---|
| 13 | `else` | 供給に余裕 |
| 15 | `elif` | 合計が一致 |
| 17 | `if` | 供給不足 |

上から条件を調べ、最初に成立した条件の処理だけを実行します。ここで比較したのは合計量です。経路に上限や通行禁止がある一般の問題では、供給の合計が十分でも実行不能になることがあります。



<br>

### 解答1：費用表の添字

費用表は `[[2, 5, 6], [4, 1, 3]]` です。倉庫1の行から消費地0の列を選びます。

```python
print(cost[1][0])
```

表示は  **4**  です。単位費用4千円/tなので、この経路で1tを運ぶ費用は4千円です。



<br>

### 解答2：需要の変更

供給は $[7,8]$ t、変更後の需要は $[4,3,10]$ tです。

$$7+8=15<4+3+10=17$$

供給できる量より必要な量が2t多いため、すべての需要を満たせません。

状態は  **`Infeasible`（実行不能）**  です。

まだ `Optimal` と表示される場合は、`changed_demand` の最後の6を10へ変更し、そのセル全体を実行したか確認してください。



<br>

### 解答3：目標点の平行移動

元の目標点は $(3,2),(3,4),(2,4.5)$ です。各目標点の横座標に1を加えると、$(4,2),(4,4),(3,4.5)$ になります。

| 条件 | 最良の回転候補 | 横の移動量 | 縦の移動量 | ずれの合計 |
|:---|---:|---:|---:|---:|
| 元の目標点 | 90° | 3 | 2 | 0.5 |
| 横へ1動かした目標点 | 90° | 4 | 2 | 0.5 |

 **横の移動量だけが1増えます。**  元の配置全体を目標点と同じ方向へ1動かせば、各点と目標点のずれは変わりません。

元の4候補のずれは、0°が5、90°が0.5、180°が5.5、270°が6.5です。



<br>

### 解答4：輸送条件の変更（任意）

入力するリストは、`trial_supply = [8, 9]`、`trial_demand = [4, 5, 6]` です。費用表は `[[2, 5, 6], [4, 1, 3]]` のままです。

- 状態：`Optimal`
- 倉庫0から消費地0・1・2への輸送量：$[4,0,2]$ t
- 倉庫1から消費地0・1・2への輸送量：$[0,5,4]$ t
- 各消費地への到着量：$[4,5,6]$ t
- 供給の残量：$[2,0]$ t

費用は次のように確かめられます。

$$2\times4+6\times2+1\times5+3\times4=37\quad\text{（千円）}$$

さらに消費地2の需要を10tに変えると、全需要19tが供給17tを超えるため、実行不能になります。


<br>

### 解答5：需要と供給の変更

倉庫・消費地の位置と単位費用を固定し、倍率だけを変更した結果です。費用は円単位に四捨五入しています。

| 条件 | 総供給上限（t） | 総需要（t） | 最小の総費用（円） |
|---|---:|---:|---:|
| 初期設定：両方1.0 | 10,000 | 7,915 | 602,850 |
| 需要1.1、供給1.0 | 10,000 | 8,706.5 | 707,741 |
| 需要1.0、供給0.9 | 9,000 | 7,915 | 648,678 |

どちらの変更後も状態は **Optimal** 、供給上限・需要・非負条件の確認はすべて **True** です。

需要を増やすと配送する量が増え、供給上限を減らすと利用できる配送の組合せが限られます。今回の2通りでは、どちらも最小費用が初期設定より増えます。

「初期設定からの増減」は、Aが **+104,891円**、Bが **+45,828円** です。表示した円単位の金額どうしの差を示しています。

④の2枚目では左の初期設定が固定され、右に今回の設定の結果が出ます。点の位置は同じですが、倉庫から消費地への配送経路や輸送量が変わります。線の太さは左右・条件間で同じ基準です。

「今回の最小の総費用」が初期設定のままなら、倍率を変えた後に①〜④を順に実行したか確認してください。総需要が総供給上限を超えた場合には、すべての需要を満たす解はありません。

参考：[PuLP公式の輸送問題の例](https://coin-or.github.io/pulp/CaseStudies/a_transportation_problem.html)。

<br>
